In [1]:
import pandas as pd 
import os
import json
import mne

os.chdir('..')

In [2]:
# Get paths from all generated experiment seqs
dir_path = "./Datasets/Target_Experiments_seq"
paths = sorted([
    os.path.join(dir_path, f)
    for f in os.listdir(dir_path)
    if f.endswith(".json") and os.path.isfile(os.path.join(dir_path, f))
])

# Or get paths from only recorded experiments
# paths = ...

len(paths)

74

In [3]:
import json
import os
from collections import Counter

pattern_counts = Counter()

for experiment_path in paths:
    with open(experiment_path, 'r') as f:
        experiment_seq = json.load(f)

    for block in experiment_seq.values():
        if block.get("type") == "pattern":
            pattern_id = block["content"].get("pattern_id")
            if pattern_id is not None and pattern_id <= 100:
                pattern_counts[pattern_id] += 1

# Вывод отсортированного результата по убыванию количества
for pid, count in sorted(pattern_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"pattern_id {pid}: {count} раз(а)")


pattern_id 10: 35 раз(а)
pattern_id 12: 35 раз(а)
pattern_id 5: 35 раз(а)
pattern_id 2: 35 раз(а)
pattern_id 4: 34 раз(а)
pattern_id 8: 34 раз(а)
pattern_id 3: 34 раз(а)
pattern_id 1: 34 раз(а)
pattern_id 11: 34 раз(а)
pattern_id 0: 34 раз(а)
pattern_id 6: 34 раз(а)
pattern_id 7: 33 раз(а)
pattern_id 9: 33 раз(а)


In [26]:
import json
from pathlib import Path

fp = Path("./Datasets/Target_Experiments_seq/experiment_26.json")
with fp.open("r", encoding="utf-8") as f:
    experiment_seq = json.load(f)

blocks = list(experiment_seq.values())

pattern_positions = [
    (i, b.get("content", {}).get("pattern_id"))
    for i, b in enumerate(blocks)
    if b.get("type") == "pattern" and b.get("content", {}).get("pattern_id") is not None and b.get("content", {}).get("pattern_id") <= 100
]

print("Найденные pattern-блоки (index, pattern_id):")
print(pattern_positions)

results = []
skip = 1  # <-- изменил: пропускать первый command (а не первые 3)
take = 7

for j in range(0, len(pattern_positions), 3):
    trip = pattern_positions[j:j+3]
    if len(trip) < 3:
        break
    idxs = [t[0] for t in trip]
    pids = [t[1] for t in trip]
    third_idx = idxs[-1]
    
    # Найдём следующие 10 command-блоков после третьего pattern
    commands = []
    k = third_idx + 1
    while k < len(blocks) and len(commands) < 10:
        if blocks[k].get("type") == "command":
            commands.append(blocks[k])
        k += 1

    print(f"\nTriplet {j//3 + 1}: pattern_ids={pids}, third_block_index={third_idx}, found {len(commands)} command-block(s) after third pattern")
    if len(commands) < 10:
        print("  WARNING: найдено меньше 10 command-блоков — используем доступные.")

    target_cmds = commands[skip:skip + take]
    states = [cmd.get("content", {}).get("state") for cmd in target_cmds]
    mapped = [ (pids[s-1] if s in (1,2,3) else None) for s in states ]
    print("  states of used commands ->", states)
    print("  mapped pattern_ids       ->", mapped)
    results.extend(mapped)

print("\nИтоговый массив для файла:", fp)
print(results)
print("Длина:", len(results))


Найденные pattern-блоки (index, pattern_id):
[(1, 11), (3, 5), (5, 0), (29, 1), (31, 4), (33, 7)]

Triplet 1: pattern_ids=[11, 5, 0], third_block_index=5, found 10 command-block(s) after third pattern
  states of used commands -> [2, 3, 3, 3, 1, 3, 2]
  mapped pattern_ids       -> [5, 0, 0, 0, 11, 0, 5]

Triplet 2: pattern_ids=[1, 4, 7], third_block_index=33, found 10 command-block(s) after third pattern
  states of used commands -> [1, 3, 1, 1, 1, 2, 3]
  mapped pattern_ids       -> [1, 7, 1, 1, 1, 4, 7]

Итоговый массив для файла: Datasets/Target_Experiments_seq/experiment_26.json
[5, 0, 0, 0, 11, 0, 5, 1, 7, 1, 1, 1, 4, 7]
Длина: 14
